# Tech Salary Prediction Pipeline (India)

## Project Objective
Build a machine learning pipeline to predict tech salaries in India. This notebook implements:
1. Data cleaning and preprocessing for a large messy dataset (~1.1 Lakh rows).
2. Feature engineering (extracting binary skill indicators).
3. Scikit-learn Pipeline incorporating standard scaling, one-hot encoding, and passthrough flags.
4. Training and comparing 4 models: ElasticNet, Random Forest, XGBoost, and CatBoost.
5. Hyperparameter tuning using RandomizedSearchCV for the best 2 models.
6. Saving the best tuned pipeline for deployment.


## 1. Import Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib
import os

## 2. Load Dataset

In [4]:
# load dataset
df = pd.read_csv('https://raw.githubusercontent.com/goyashek/Tech-Salary-Advisor/refs/heads/main/data/salary_dataset_dirty.csv')
df.head()

,Job_Title,Experience_Years,Education_Level,Location,Skills,Salary_INR
0,AI Researcher,1.2,NaN,noida,"Python,JavaScript,SQL,Spark,Docker,TensorFlow",2159338.0
1,QA,4.3,phd,Noida,"Java, JavaScript, Go, Docker, Kubernetes, Tens...",1084669.0
2,backend engineer,3.6,M.S.,NaN,"Python,JavaScript,System Design,Agile",1040748.0
3,data scientist,0.5,PhD,Bangalore,"Python, Java, SQL, AWS, React",1939103.0
4,cv engineer,3.9,masters,Noida,"Python, C++, SQL, PyTorch, React, Agile",1723630.0


In [5]:
# check dataset shape
df.shape

(110000, 6)

In [7]:
# get summary
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110000 entries, 0 to 109999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Job_Title         110000 non-null  str    
 1   Experience_Years  104425 non-null  float64
 2   Education_Level   104566 non-null  str    
 3   Location          104516 non-null  str    
 4   Skills            104432 non-null  str    
 5   Salary_INR        107735 non-null  float64
dtypes: float64(2), str(4)
memory usage: 12.3 MB


## 3. Data Cleaning and Handling Missing Values

In [8]:
# check for any missing values
df.isnull().sum()

Job_Title              0
Experience_Years    5575
Education_Level     5434
Location            5484
Skills              5568
Salary_INR          2265
dtype: int64

In [9]:
# drop rows where target variable Salary_INR is missing
df = df.dropna(subset=['Salary_INR'])
df.shape

(107735, 6)

In [10]:
# Impute missing Experience_Years with median
exp_median = df['Experience_Years'].median()
df['Experience_Years'] = df['Experience_Years'].fillna(exp_median)
print(f"Imputed missing experience with median value: {exp_median}")

Imputed missing experience with median value: 4.1


In [11]:
# Impute missing categorical columns with mode
df['Education_Level'] = df['Education_Level'].fillna(df['Education_Level'].mode()[0])
df['Location'] = df['Location'].fillna(df['Location'].mode()[0])
# Impute missing Skills with a default fallback
df['Skills'] = df['Skills'].fillna('Python, SQL')

In [12]:
# verify that no missing values remain
df.isnull().sum()

Job_Title           0
Experience_Years    0
Education_Level     0
Location            0
Skills              0
Salary_INR          0
dtype: int64

## 4. Standardizing Inconsistent Categorical Values

In [13]:
# clean up job titles
def clean_job_title(title):
    title = str(title).strip().lower()
    if 'data scientist' in title:
        return 'Data Scientist'
    elif 'ml engineer' in title or 'machine learning' in title or 'ai engineer' in title:
        return 'Machine Learning Engineer'
    elif 'ai researcher' in title:
        return 'AI Researcher'
    elif 'nlp' in title:
        return 'NLP Engineer'
    elif 'cv engineer' in title or 'computer vision' in title:
        return 'Computer Vision Engineer'
    elif 'dl engineer' in title or 'deep learning' in title:
        return 'Deep Learning Engineer'
    elif 'frontend' in title:
        return 'Frontend Developer'
    elif 'backend' in title:
        return 'Backend Developer'
    elif 'fullstack' in title or 'full stack' in title:
        return 'Full Stack Developer'
    elif 'devops' in title:
        return 'DevOps Engineer'
    elif 'data engineer' in title:
        return 'Data Engineer'
    elif 'qa' in title:
        return 'QA Engineer'
    elif 'product' in title or 'pm' in title:
        return 'Product Manager'
    else:
        return 'Software Engineer'

df['Job_Title'] = df['Job_Title'].apply(clean_job_title)
df['Job_Title'].value_counts()

Job_Title
Machine Learning Engineer    13490
Software Engineer            10140
Data Scientist               10023
Deep Learning Engineer        6855
Computer Vision Engineer      6814
Data Engineer                 6814
Full Stack Developer          6767
AI Researcher                 6736
NLP Engineer                  6716
QA Engineer                   6702
DevOps Engineer               6675
Frontend Developer            6675
Backend Developer             6673
Product Manager               6655
Name: count, dtype: int64

In [14]:
# Function to standardize location names
def clean_location(city):
    city = str(city).strip().lower()
    if 'bangalore' in city:
        return 'Bangalore'
    elif 'mumbai' in city:
        return 'Mumbai'
    elif 'delhi' in city:
        return 'Delhi NCR'
    elif 'noida' in city:
        return 'Noida'
    elif 'hyderabad' in city:
        return 'Hyderabad'
    elif 'pune' in city:
        return 'Pune'
    elif 'chennai' in city:
        return 'Chennai'
    else:
        return 'Bangalore'

df['Location'] = df['Location'].apply(clean_location)
df['Location'].value_counts()

Location
Bangalore    20031
Pune         14817
Chennai      14667
Delhi NCR    14601
Noida        14560
Mumbai       14548
Hyderabad    14511
Name: count, dtype: int64

In [15]:
# Clean up education categories
def clean_education(edu):
    edu = str(edu).strip().lower()
    if 'master' in edu or 'mtech' in edu or 'm.tech' in edu or 'ms' in edu or 'm.s.' in edu:
        return "Master's"
    elif 'phd' in edu or 'doctor' in edu:
        return 'PhD'
    else:
        return "Bachelor's"

df['Education_Level'] = df['Education_Level'].apply(clean_education)
df['Education_Level'].value_counts()

Education_Level
Bachelor's    44711
Master's      39427
PhD           23597
Name: count, dtype: int64

## 5. Feature Engineering (Skills Parsing)

In [16]:
# Skills flags
all_skills = [
    "Python", "Java", "C++", "JavaScript", "Go", 
    "SQL", "Spark", "AWS", "Docker", "Kubernetes", 
    "PyTorch", "TensorFlow", "React", "System Design", "Agile"
]

# Create binary columns
for skill in all_skills:
    df[skill] = df['Skills'].apply(lambda x: 1 if skill.lower() in str(x).lower() else 0)

# Drop raw skills
df = df.drop(columns=['Skills'])
df.head()

,Job_Title,Experience_Years,Education_Level,Location,Salary_INR,Python,Java,C++,JavaScript,Go,SQL,Spark,AWS,Docker,Kubernetes,PyTorch,TensorFlow,React,System Design,Agile
0,AI Researcher,1.2,Bachelor's,Noida,2159338.0,1,1,0,1,0,1,1,0,1,0,0,1,0,0,0
1,QA Engineer,4.3,PhD,Noida,1084669.0,0,1,0,1,1,0,0,0,1,1,0,1,1,1,1
2,Backend Developer,3.6,Master's,Bangalore,1040748.0,1,1,0,1,0,0,0,0,0,0,0,0,0,1,1
3,Data Scientist,0.5,PhD,Bangalore,1939103.0,1,1,0,0,0,1,0,1,0,0,0,0,1,0,0
4,Computer Vision Engineer,3.9,Master's,Noida,1723630.0,1,0,1,0,0,1,0,0,0,0,1,0,1,0,1


## 6. Pipeline Construction and Data Splitting

In [17]:
# split target and features
X = df.drop(columns=['Salary_INR'])
y = df['Salary_INR']

# split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

Train size: (86188, 19)
Test size: (21547, 19)


In [20]:
# identify column types for the transformer
numeric_cols = ['Experience_Years']
categorical_cols = ['Job_Title', 'Location', 'Education_Level']

# construct the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

## 7. Model Training and Comparison

In [21]:
# Define 4 models to compare
models = {
    'ElasticNet':ElasticNet(alpha=0.4, l1_ratio=0.7, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, n_jobs=-1),
    "CatBoost": CatBoostRegressor(iterations=100,learning_rate=0.05,depth=6,loss_function='RMSE',eval_metric='RMSE',random_seed=42, verbose=False)    
    #"Support Vector Regression": SVR(kernel='rbf', epsilon=0.1, C=1.0)
}

trained_pipelines = {}
model_predictions = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline
    model_predictions[name] = pipeline.predict(X_test)
    print(f"{name} training complete! \n")

Training ElasticNet...
ElasticNet training complete! 

Training Random Forest...
Random Forest training complete! 

Training XGBoost...
XGBoost training complete! 

Training CatBoost...
CatBoost training complete! 



## 8. Model Evaluation

In [22]:
metrics_summary = {}

for name in models.keys():
    preds = model_predictions[name]
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    metrics_summary[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    
    print(f"=== {name} Metrics ===")
    print(f"MAE:  Rs. {mae:,.2f}")
    print(f"RMSE: Rs. {rmse:,.2f}")
    print(f"R2:   {r2:.4f}\n")

=== ElasticNet Metrics ===
MAE:  Rs. 243,005.96
RMSE: Rs. 319,254.27
R2:   0.7283

=== Random Forest Metrics ===
MAE:  Rs. 196,974.90
RMSE: Rs. 259,184.51
R2:   0.8209

=== XGBoost Metrics ===
MAE:  Rs. 164,437.08
RMSE: Rs. 221,085.60
R2:   0.8697

=== CatBoost Metrics ===
MAE:  Rs. 180,936.73
RMSE: Rs. 240,437.76
R2:   0.8459



## 9. Hyperparameter Tuning for Best 2 Models

In [24]:
# To speed up tuning on ~80k rows, you can maybe select a subset of it my modifying 'size' , we are taking full dataset here
np.random.seed(42)
subset_indices = np.random.choice(len(X_train), size=86188, replace=False)
X_train_sub = X_train.iloc[subset_indices]
y_train_sub = y_train.iloc[subset_indices]

print(f"Tuning subset sizes: {X_train_sub.shape}")

Tuning subset sizes: (86188, 19)


In [25]:
# tune XGBoost
xgb_param_dist = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.05, 0.1, 0.2]
}

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, n_jobs=-1))
])

xgb_search = RandomizedSearchCV(xgb_pipeline, param_distributions=xgb_param_dist, n_iter=8, cv=3, scoring='r2', random_state=42, n_jobs=-1,verbose=True)
xgb_search.fit(X_train_sub, y_train_sub)
print(f"Best XGBoost Params: {xgb_search.best_params_}")
print(f"Best XGBoost CV R2 Score: {xgb_search.best_score_:.4f}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best XGBoost Params: {'model__n_estimators': 100, 'model__max_depth': 6, 'model__learning_rate': 0.1}
Best XGBoost CV R2 Score: 0.8680


In [28]:
cat_param_dist = {
    'model__iterations': [600, 800, 1000],
    'model__learning_rate': [0.03, 0.05, 0.07],
    'model__depth': [4, 5, 6],
    'model__l2_leaf_reg': [3, 4, 5]
}
cat_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', CatBoostRegressor(loss_function='RMSE',random_state=42, verbose=False))
])

cat_search = RandomizedSearchCV(cat_pipeline,param_distributions=cat_param_dist,n_iter=10,cv=3,scoring='r2',random_state=42,n_jobs=-1, verbose=True)

cat_search.fit(X_train_sub, y_train_sub)

print(f"Best CatBoost Params: {cat_search.best_params_}")
print(f"Best CatBoost CV R2 Score: {cat_search.best_score_:.4f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best CatBoost Params: {'model__learning_rate': 0.05, 'model__l2_leaf_reg': 5, 'model__iterations': 800, 'model__depth': 6}
Best CatBoost CV R2 Score: 0.8722


In [30]:
# Function to display model performance
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"=== {name} Metrics ===")
    print(f"Mean Absolute Error (MAE): Rs. {mae:,.2f}")
    print(f"Root Mean Squared Error (RMSE): Rs. {rmse:,.2f}")
    print(f"R2 Score: {r2:.4f}\n")
    return mae, r2

# Evaluate tuned models on full test set
best_xgb_model = xgb_search.best_estimator_
best_cb_model = cat_search.best_estimator_

tuned_xgb_preds = best_xgb_model.predict(X_test)
tuned_cb_preds = best_cb_model.predict(X_test)

tuned_xgb_mae, tuned_xgb_r2 = evaluate_model("Tuned XGBoost", y_test, tuned_xgb_preds)
tuned_cb_mae, tuned_cb_r2 = evaluate_model("Tuned CatBoost", y_test, tuned_cb_preds)

=== Tuned XGBoost Metrics ===
Mean Absolute Error (MAE): Rs. 163,186.27
Root Mean Squared Error (RMSE): Rs. 219,538.55
R2 Score: 0.8715

=== Tuned CatBoost Metrics ===
Mean Absolute Error (MAE): Rs. 160,258.95
Root Mean Squared Error (RMSE): Rs. 216,295.24
R2 Score: 0.8753



## 10. Save Tuned Pipeline and Metadata

In [31]:
# Select final model
final_pipeline = best_xgb_model if tuned_xgb_r2 >= tuned_cb_r2 else best_cb_model
final_name = "Tuned XGBoost" if tuned_xgb_r2 >= tuned_cb_r2 else "Tuned CatBoost"
final_mae = tuned_xgb_mae if tuned_xgb_r2 >= tuned_cb_r2 else tuned_cb_mae
final_r2 = tuned_xgb_r2 if tuned_xgb_r2 >= tuned_cb_r2 else tuned_cb_r2

print(f"Selected Final Model Pipeline: {final_name}")

Selected Final Model Pipeline: Tuned CatBoost


In [32]:
# Save the final tuned pipeline
os.makedirs('../streamlit/models', exist_ok=True)
joblib.dump(final_pipeline, '../streamlit/models/salary_model.pkl')

# Create lookup options metadata
metadata = {
    'feature_columns': list(X.columns),
    'all_skills': all_skills,
    'median_exp': exp_median,
    'mae': float(final_mae),
    'r2': float(final_r2),
    'model_name': final_name,
    'job_titles': ['Software Engineer', 'Frontend Developer', 'Backend Developer', 'Full Stack Developer', 
                   'Data Scientist', 'Machine Learning Engineer', 'AI Researcher', 'NLP Engineer', 
                   'Computer Vision Engineer', 'Deep Learning Engineer', 'DevOps Engineer', 'Data Engineer', 
                   'QA Engineer', 'Product Manager'],
    'locations': ['Bangalore', 'Mumbai', 'Delhi NCR', 'Noida', 'Hyderabad', 'Pune', 'Chennai'],
    'education_levels': ["Bachelor's", "Master's", "PhD"]
}

joblib.dump(metadata, '../streamlit/models/metadata.pkl')
print("Exported pkl")

Exported pkl
